# 🚀 LLM-XAI Benchmark Runner (Google Colab Pro Edition)
### Badanie stabilności (RQ1), wierności (RQ2), ich relacji (RQ3) oraz efektywności (RQ4) dla Causal LM na GPU (L4 / A100 / T4)

**Autor:** inż. Jakub Stachyra | **Promotor:** dr Robert Małysz  
*Wydział Matematyki i Nauk Informacyjnych, Politechnika Warszawska*

---
Ten notatnik jest zoptymalizowanym punktem wejścia do uruchamiania pełnego potoku badawczego w chmurze Google Colab Pro.
Wykorzystuje **odporny checkpointing co 10 próbek** oraz automatyczną synchronizację z Dyskiem Google.

## 1. Weryfikacja akceleratora GPU (Zalecane: L4 lub A100)

In [ ]:
# Sprawdzenie dostępnego GPU i pamięci VRAM
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {device_name} ({vram_gb:.1f} GB VRAM)")
    print(f"BFloat16 supported: {torch.cuda.is_bf16_supported()}")
else:
    print("OSTRZEŻENIE: Brak GPU! Zmień typ środowiska wykonawczego (Runtime -> Change runtime type -> T4/L4/A100).")

## 2. Podmontowanie Dysku Google (Trwały zapis checkpointów)
Dzięki zapisowi na Google Drive nawet w razie zerwania połączenia lub wygaśnięcia sesji Colaba **nie tracisz ani jednej policzonej próbki**!

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Tworzymy katalog na Dysku Google na wyniki magisterki
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/llm_xai_results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Katalog na Dysku Google gotowy: {DRIVE_RESULTS_DIR}")

## 3. Pobranie kodu i instalacja zależności

## 3.5. Badanie pilotażowe zbieżności i budżetu SHAP (Ablation Study)
Przed uruchomieniem pełnego benchmarku wyznaczamy empiryczny punkt zbieżności estymatora `PermutationExplainer` dla siatki `max_evals` $\in \{50, 100, 250, 500, 1000\}$.
Generuje wykres krzywej Pareto (stabilność i powtarzalność vs czas wykonania) do Rozdziału 3 pracy magisterskiej.

In [ ]:
#@title Uruchomienie badania zbieżności SHAP
from IPython.display import Image, display
import os

samples_shap_study = 15 #@param {type:"integer"}
grid_shap = "50,100,250,500,1000" #@param {type:"string"}
output_shap_study = f"{DRIVE_RESULTS_DIR}/shap_convergence"

!python scripts/audit_shap_convergence.py \
    --model microsoft/phi-2 \
    --samples {samples_shap_study} \
    --grid {grid_shap} \
    --output-dir {output_shap_study}

# Wyświetlenie wykresu zbieżności
shap_plot = f"{output_shap_study}/plots/shap_convergence_curve.png"
if os.path.exists(shap_plot):
    display(Image(filename=shap_plot))

In [ ]:
#@title Konfiguracja repozytorium
import os

# Jeśli klonujesz ze swojego GitHub:
GITHUB_REPO_URL = "https://github.com/jakubstachyra/llm-xai.git" #@param {type:"string"}

if not os.path.exists("/content/llm-xai"):
    !git clone {GITHUB_REPO_URL} /content/llm-xai

%cd /content/llm-xai

# Instalacja wymaganych bibliotek
!pip install -q transformers accelerate datasets shap scikit-learn matplotlib pandas scipy

## 4. Pytanie Badawcze 1 (RQ1 - Badanie Stabilności)
Ocena odporności rankingów atrybucji na 16 neutralnych perturbacji tekstu (korelacja Spearmana $\rho$, Kendall $\tau$, Top-$K$ Jaccard).

In [ ]:
#@title Konfiguracja RQ1 (Stabilność)
model_rq1 = "microsoft/phi-2" #@param ["microsoft/phi-2", "Qwen/Qwen2.5-7B", "meta-llama/Llama-3.1-8B"]
samples_rq1 = 400 #@param {type:"integer"}
methods_rq1 = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}
shap_evals_rq1 = 500 #@param {type:"integer"}
ig_steps_rq1 = 20 #@param {type:"integer"}

output_rq1 = f"{DRIVE_RESULTS_DIR}/rq1_stability"

!python scripts/run_rq1.py \
    --model {model_rq1} \
    --samples {samples_rq1} \
    --methods {methods_rq1} \
    --max-evals {shap_evals_rq1} \
    --ig-steps {ig_steps_rq1} \
    --output-dir {output_rq1}

## 5. Pytanie Badawcze 2 (RQ2 - Badanie Wierności / Faithfulness)
Pomiar spadku prawdopodobieństwa predykcji po usunięciu najważniejszych słów ($C(k)$ Comprehensiveness, $U(k)$ Sufficiency, $\mathrm{AOPC}$, luka informacyjna $G$ ponad Random Baseline).

In [ ]:
#@title Konfiguracja RQ2 (Wierność)
model_rq2 = "microsoft/phi-2" #@param ["microsoft/phi-2", "Qwen/Qwen2.5-7B", "meta-llama/Llama-3.1-8B"]
samples_rq2 = 400 #@param {type:"integer"}
operator_rq2 = "deletion" #@param ["deletion", "embedding_mask"]
methods_rq2 = "occlusion,grad_x_input,ig,lime,shap" #@param {type:"string"}
shap_evals_rq2 = 500 #@param {type:"integer"}
random_trials_rq2 = 20 #@param {type:"integer"}

output_rq2 = f"{DRIVE_RESULTS_DIR}/rq2_faithfulness_{operator_rq2}"

!python scripts/run_rq2.py \
    --model {model_rq2} \
    --samples {samples_rq2} \
    --operator {operator_rq2} \
    --methods {methods_rq2} \
    --max-evals-shap {shap_evals_rq2} \
    --random-trials {random_trials_rq2} \
    --output-dir {output_rq2}

## 6. Pytanie Badawcze 3 (RQ3 - Analiza Łączona: Stabilność vs Wierność)
Wyliczenie korelacji między stabilnością rankingów a wiernością predykcyjną na tych samych recenzjach, wygenerowanie wykresu punktowego oraz tabeli LaTeX.

In [ ]:
import os
from IPython.display import Image, display
from src.experiments.link_rq1_rq2 import generate_joint_rq1_rq2_analysis

output_rq3 = f"{DRIVE_RESULTS_DIR}/rq3_joint_analysis"
joint_df, tex_table = generate_joint_rq1_rq2_analysis(
    rq1_summary_path=f"{output_rq1}/rq1_stability_summary.csv",
    rq2_summary_path=f"{output_rq2}/rq2_faithfulness_summary.csv",
    output_dir=output_rq3
)

print("\n" + "="*60)
print("  WYGENEROWANA TABELA LATEX DO PRACY MAGISTERSKIEJ (ROZDZIAŁ 4):")
print("="*60)
print(tex_table)

# Wyświetlenie wykresu relacji bezpośrednio w notatniku
plot_file = os.path.join(output_rq3, "rq3_stability_vs_faithfulness.png")
if os.path.exists(plot_file):
    display(Image(filename=plot_file))

## 7. Podgląd wygenerowanych rycin dla RQ1 i RQ2

In [ ]:
from IPython.display import Image, display

# Krzywe Comprehensiveness C(k) dla RQ2
rq2_plot = f"{output_rq2}/plots/rq2_comprehensiveness_curves.png"
if os.path.exists(rq2_plot):
    print("RQ2: Krzywe Comprehensiveness C(k) vs Random Baseline:")
    display(Image(filename=rq2_plot))

# Porównanie AOPC_C vs AOPC_U dla RQ2
rq2_aopc_plot = f"{output_rq2}/plots/rq2_aopc_comparison.png"
if os.path.exists(rq2_aopc_plot):
    print("RQ2: Porównanie AOPC:")
    display(Image(filename=rq2_aopc_plot))

# Rozkład Spearmana dla RQ1
rq1_plot = f"{output_rq1}/plots/rq1_spearman_distribution.png"
if os.path.exists(rq1_plot):
    print("RQ1: Rozkłady stabilności Spearmana:")
    display(Image(filename=rq1_plot))